In [1]:
!pip install -U transformers peft datasets bitsandbytes trl accelerate qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 109.2 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstal

In [2]:
import torch
import json
import os
import gc
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset, DatasetDict
from qwen_vl_utils import process_vision_info
from sklearn.model_selection import train_test_split

MODEL_ID = "Qwen/Qwen2-VL-7B-Instruct"

# 1. Quantization Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# 2. Load Model & Processor
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256*28*28, max_pixels=512*28*28)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

...

chat_template.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
def prepare_map_dataset(mcq_json_path, image_folder):
    with open(mcq_json_path, 'r') as f:
        mcqs = json.load(f)
    
    formatted_data = []
    for item in mcqs:
        img_filename = os.path.basename(item['image_path'])
        img_path = os.path.join(image_folder, img_filename)
        cls = str(item.get("classification", "unknown")).strip().lower()

        if cls == "unanswerable":
            continue

        prompt = prompt = (
            "You are a highly intelligent assistant. "
            "Based on the given image, answer the multiple-choice question by selecting the correct option.\n\n"
            "Question:\n" + item["question"] + "\n\n"
            "Options:\n" + item["options"] + "\n"
            "\nSelect the best option by choosing its number."
        )
        
        if os.path.exists(img_path):
            formatted_data.append({
                "image": img_path,
                "question": prompt,
                "answer": str(item['answer']),
                "classification": cls
            })
    
    return formatted_data

IMAGE_FOLDER = "/kaggle/input/datasets/ekramulhaqueamin/fulldataset/dataset/images"

JSON_FILES = {
    "counting": "/kaggle/input/datasets/ekramulhaqueamin/fulldataset/dataset/output_mcqs/counting_mcqs.json",
    "nearby":   "/kaggle/input/datasets/ekramulhaqueamin/fulldataset/dataset/output_mcqs/nearby_mcqs.json",
    "poi":      "/kaggle/input/datasets/ekramulhaqueamin/fulldataset/dataset/output_mcqs/poi_mcqs.json",
    "routing":  "/kaggle/input/datasets/ekramulhaqueamin/fulldataset/dataset/output_mcqs/routing_mcqs.json",
}

# Load and split each category separately (same 70/15/15 ratio)
train_list, val_list, test_list = [], [], []
RAW_LIST = []

for cat, json_path in JSON_FILES.items():
    cat_data = prepare_map_dataset(json_path, IMAGE_FOLDER)
    RAW_LIST.extend(cat_data)

    cat_train, cat_temp = train_test_split(cat_data, test_size=0.3, random_state=42)
    cat_val, cat_test   = train_test_split(cat_temp, test_size=0.5, random_state=42)

    train_list.extend(cat_train)
    val_list.extend(cat_val)
    test_list.extend(cat_test)

    print(f"{cat:10s} → total: {len(cat_data)} | train: {len(cat_train)} | val: {len(cat_val)} | test: {len(cat_test)}")

dataset = DatasetDict({
    "train":      Dataset.from_list(train_list),
    "validation": Dataset.from_list(val_list),
    "test":       Dataset.from_list(test_list)
})

print(f"\nTotal RAW_LIST: {len(RAW_LIST)}")
print({k: len(v) for k, v in dataset.items()})

counting   → total: 745 | train: 521 | val: 112 | test: 112
nearby     → total: 547 | train: 382 | val: 82 | test: 83
poi        → total: 784 | train: 548 | val: 118 | test: 118
routing    → total: 880 | train: 616 | val: 132 | test: 132

Total RAW_LIST: 2956
{'train': 2067, 'validation': 444, 'test': 445}


In [5]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    modules_to_save=["lm_head"] 
)
model = get_peft_model(model, lora_config)

def collate_fn(examples):
    messages = []
    for ex in examples:
        messages.append([
            {"role": "user", "content": [{"type": "image", "image": ex["image"]}, {"type": "text", "text": ex["question"]}]},
            {"role": "assistant", "content": [{"type": "text", "text": ex["answer"]}]}
        ])
    
    texts = [processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=False) for msg in messages]
    image_inputs, _ = process_vision_info(messages)
    
    inputs = processor(text=texts, images=image_inputs, padding=True, return_tensors="pt")
    inputs["labels"] = inputs["input_ids"].clone()
    inputs["labels"][inputs["labels"] == processor.tokenizer.pad_token_id] = -100
    
    return inputs

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


In [6]:
training_args = TrainingArguments(
    output_dir="qwen2-vl-maps-output",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    learning_rate=1e-4,
    num_train_epochs=10,
    bf16=True, # Use bf16 if supported, else use fp16=True
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=1,  # Keep only the best checkpoint
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    remove_unused_columns=False
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=collate_fn,
)

trainer.train()

# Model now holds the best checkpoint based on validation loss

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
250,0.098615,0.087337
500,0.068843,0.086574
750,0.040204,0.096634
1000,0.037937,0.096777
1250,0.035772,0.098407
1500,0.030321,0.102838
1750,0.033207,0.102866
2000,0.025913,0.104182
2250,0.024943,0.103491
2500,0.026956,0.104079


TrainOutput(global_step=2590, training_loss=0.09762407423043803, metrics={'train_runtime': 8365.2066, 'train_samples_per_second': 2.471, 'train_steps_per_second': 0.31, 'total_flos': 6.201011426750669e+17, 'train_loss': 0.09762407423043803})

In [7]:
# ─── FINE-TUNED MODEL EVALUATION ──────────────────────────────────────────────
from collections import defaultdict

EXTERNAL_JSON   = "/kaggle/input/datasets/ekramulhaqueamin/datset/dataset/dataset.json"
EXTERNAL_IMAGES = "/kaggle/input/datasets/ekramulhaqueamin/datset/dataset/VData"

def load_external_dataset(mcq_json_path, image_folder):
    with open(mcq_json_path, 'r') as f:
        mcqs = json.load(f)
    data = []
    skipped = 0
    for item in mcqs:
        cls = str(item.get("classification", "unknown")).strip().lower()
        if cls == "unanswerable":
            skipped += 1
            continue
        img_filename = os.path.basename(item['image_path'])
        img_path = os.path.join(image_folder, img_filename)
        if not os.path.exists(img_path):
            skipped += 1
            continue
        prompt = (
            "You are a highly intelligent assistant. "
            "Based on the given image, answer the multiple-choice question by selecting the correct option.\n\n"
            "Question:\n" + item["question"] + "\n\n"
            "Options:\n" + item["options"] + "\n"
            "\nSelect the best option by choosing its number."
        )
        data.append({
            "image": img_path,
            "question": prompt,
            "answer": str(item['answer']),
            "classification": cls
        })
    print(f"Loaded: {len(data)} | Skipped: {skipped}")
    return data

def run_evaluation(data, mdl, proc, label="", is_peft=False):
    mdl.eval()
    if hasattr(mdl, "gradient_checkpointing_disable"):
        mdl.gradient_checkpointing_disable()
    mdl.config.use_cache = True
    if is_peft and hasattr(mdl, 'base_model') and hasattr(mdl.base_model.model, 'lm_head'):
        mdl.base_model.model.lm_head = mdl.base_model.model.lm_head.to(torch.float32)

    device = next(mdl.parameters()).device
    total = len(data)
    overall_correct = 0
    category_stats = defaultdict(lambda: {"correct": 0, "total": 0})

    print(f"\nEvaluating {label} ({total} samples)...")

    for i in range(total):
        sample = data[i]
        cat = sample.get("classification", "unknown") if isinstance(sample, dict) else sample["classification"]

        messages = [[{
            "role": "user",
            "content": [
                {"type": "image", "image": sample["image"]},
                {"type": "text",  "text":  sample["question"]}
            ]
        }]]

        text = [proc.apply_chat_template(msg, tokenize=False, add_generation_prompt=True) for msg in messages]
        image_inputs, _ = process_vision_info(messages)
        inputs = proc(text=text, images=image_inputs, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            generated_ids = mdl.generate(**inputs, max_new_tokens=10)
            generated_ids = [out[len(inp):] for inp, out in zip(inputs["input_ids"], generated_ids)]
            response = proc.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

        pred   = response.strip()
        actual = str(sample["answer"]).strip()

        is_correct = actual in pred or pred in actual
        if is_correct:
            overall_correct += 1
            category_stats[cat]["correct"] += 1
        category_stats[cat]["total"] += 1

        if i % 10 == 0:
            print(f"Processed {i}/{total} | pred: '{pred}' | actual: '{actual}'")

    print(f"\nOverall Accuracy ({label}): {(overall_correct/total)*100:.2f}%")
    print("\n─── Per-Category Summary ───────────────────────")
    for cat, stats in category_stats.items():
        acc = (stats["correct"] / stats["total"]) * 100
        print(f"  {cat:12s}: {acc:.2f}%  ({stats['correct']}/{stats['total']})")
    print("────────────────────────────────────────────────")
    return {cat: stats for cat, stats in category_stats.items()}, overall_correct, total

# Load external dataset
external_data = load_external_dataset(EXTERNAL_JSON, EXTERNAL_IMAGES)

# ── Fine-tuned: internal test set ─────────────────────────────────────────────
ft_internal, ft_int_correct, ft_int_total = run_evaluation(
    dataset["test"], model, processor,
    label="FINE-TUNED | Internal Test Set", is_peft=True
)

# ── Fine-tuned: external dataset ──────────────────────────────────────────────
ft_external, ft_ext_correct, ft_ext_total = run_evaluation(
    external_data, model, processor,
    label="FINE-TUNED | External Dataset", is_peft=True
)

# Free fine-tuned model before loading base
del model
gc.collect()
torch.cuda.empty_cache()
print("Fine-tuned model freed from memory.")

Loaded: 380 | Skipped: 20

Evaluating FINE-TUNED | Internal Test Set (445 samples)...
Processed 0/445 | pred: '4' | actual: '4'
Processed 10/445 | pred: '3' | actual: '3'
Processed 20/445 | pred: '2' | actual: '2'
Processed 30/445 | pred: '3' | actual: '3'
Processed 40/445 | pred: '3' | actual: '3'
Processed 50/445 | pred: '1' | actual: '1'
Processed 60/445 | pred: '3' | actual: '3'
Processed 70/445 | pred: '1' | actual: '1'
Processed 80/445 | pred: '4' | actual: '4'
Processed 90/445 | pred: '4' | actual: '4'
Processed 100/445 | pred: '1' | actual: '1'
Processed 110/445 | pred: '3' | actual: '3'
Processed 120/445 | pred: '1' | actual: '1'
Processed 130/445 | pred: '4' | actual: '2'
Processed 140/445 | pred: '4' | actual: '4'
Processed 150/445 | pred: '3' | actual: '3'
Processed 160/445 | pred: '4' | actual: '1'
Processed 170/445 | pred: '3' | actual: '2'
Processed 180/445 | pred: '1' | actual: '1'
Processed 190/445 | pred: '2' | actual: '2'
Processed 200/445 | pred: '3' | actual: '3'
P

In [8]:
# ─── BASE MODEL EVALUATION ─────────────────────────────────────────────────────
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)
base_processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256*28*28, max_pixels=512*28*28)

# ── Base: internal test set ───────────────────────────────────────────────────
base_internal, base_int_correct, base_int_total = run_evaluation(
    dataset["test"], base_model, base_processor,
    label="BASE | Internal Test Set", is_peft=False
)

# ── Base: external dataset ────────────────────────────────────────────────────
base_external, base_ext_correct, base_ext_total = run_evaluation(
    external_data, base_model, base_processor,
    label="BASE | External Dataset", is_peft=False
)

# Free base model
del base_model
del base_processor
gc.collect()
torch.cuda.empty_cache()

# ─── FINAL COMPARISON SUMMARY ──────────────────────────────────────────────────
def print_comparison(label, base_res, ft_res, base_correct, base_total, ft_correct, ft_total):
    print(f"\n─── {label} ───────────────────────────────────────")
    print(f"  {'Category':12s}  {'Base':>8}  {'Fine-Tuned':>10}  {'Delta':>8}")
    print("  " + "─" * 48)
    for cat in base_res:
        if cat not in ft_res:
            continue
        b = (base_res[cat]["correct"] / base_res[cat]["total"]) * 100
        f = (ft_res[cat]["correct"]   / ft_res[cat]["total"])   * 100
        d = f - b
        arrow = "▲" if d > 0 else "▼"
        print(f"  {cat:12s}  {b:>7.2f}%  {f:>9.2f}%  {arrow}{abs(d):>6.2f}%")
    print("  " + "─" * 48)
    b_ov = (base_correct / base_total) * 100
    f_ov = (ft_correct   / ft_total)   * 100
    d_ov = f_ov - b_ov
    arrow = "▲" if d_ov > 0 else "▼"
    print(f"  {'OVERALL':12s}  {b_ov:>7.2f}%  {f_ov:>9.2f}%  {arrow}{abs(d_ov):>6.2f}%")
    print("  " + "─" * 48)

print_comparison("Internal Test Set", base_internal, ft_internal, base_int_correct, base_int_total, ft_int_correct, ft_int_total)
print_comparison("External Dataset",  base_external, ft_external, base_ext_correct, base_ext_total, ft_ext_correct, ft_ext_total)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]


Evaluating BASE | Internal Test Set (445 samples)...
Processed 0/445 | pred: '4) 1' | actual: '4'
Processed 10/445 | pred: '2' | actual: '3'
Processed 20/445 | pred: '2) 1' | actual: '2'
Processed 30/445 | pred: '1) 2' | actual: '3'
Processed 40/445 | pred: '3) 1' | actual: '3'
Processed 50/445 | pred: '2' | actual: '1'
Processed 60/445 | pred: '3) 1' | actual: '3'
Processed 70/445 | pred: '1) 1' | actual: '1'
Processed 80/445 | pred: '3) 0' | actual: '4'
Processed 90/445 | pred: '4) 1' | actual: '4'
Processed 100/445 | pred: '2) 0' | actual: '1'
Processed 110/445 | pred: '1' | actual: '3'
Processed 120/445 | pred: '4' | actual: '1'
Processed 130/445 | pred: '4)팔당냉면 강남본점' | actual: '2'
Processed 140/445 | pred: '2' | actual: '4'
Processed 150/445 | pred: '3' | actual: '3'
Processed 160/445 | pred: '3' | actual: '1'
Processed 170/445 | pred: '2' | actual: '2'
Processed 180/445 | pred: '1' | actual: '1'
Processed 190/445 | pred: '2' | actual: '2'
Processed 200/445 | pred: '3' | actual: 